In [1]:
import pandas as pd
import polars as pl
import numpy as np
import gc
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier

In [2]:
pretrain = pl.read_parquet("../data/pretrain_part_1.parquet")

In [3]:
pretrain = pretrain.with_columns(pl.col("event_dttm").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S")\
                                 .alias("event_dttm")).with_columns(pl.col("event_dttm").dt.hour().alias("Hour"))

pre_agg = pretrain.group_by("customer_id").agg([
    pl.col("operaton_amt").mean().alias("mean_amt_pre"),
    pl.col("operaton_amt").median().alias("median_amt_pre"),
    pl.col("operaton_amt").std().alias("std_amt_pre"),
    pl.col("event_id").count().alias("ops_pre"),
    pl.col("Hour").mode().alias("most_common_hour_pre")])

pre_agg = pre_agg.with_columns(pl.col("most_common_hour_pre").list.first().alias("most_common_hour_pre"))
del pretrain
gc.collect()

8

In [4]:
train_part1 = pl.read_parquet("../data/test.parquet")

In [5]:
train_part1 = train_part1.with_columns([
    pl.col("compromised").cast(pl.Int32),
    pl.col("developer_tools").cast(pl.Int32)
])

In [6]:
train_part1.schema

Schema([('customer_id', Int64),
        ('event_id', Int64),
        ('event_dttm', String),
        ('event_type_nm', Int32),
        ('event_desc', Int32),
        ('channel_indicator_type', Int32),
        ('channel_indicator_sub_type', Int32),
        ('operaton_amt', Float64),
        ('currency_iso_cd', Int32),
        ('mcc_code', String),
        ('pos_cd', Int32),
        ('accept_language', String),
        ('browser_language', String),
        ('timezone', Int32),
        ('session_id', Int64),
        ('operating_system_type', Int32),
        ('battery', String),
        ('device_system_version', String),
        ('screen_size', String),
        ('developer_tools', Int32),
        ('phone_voip_call_state', Int32),
        ('web_rdp_connection', Int32),
        ('compromised', Int32)])

In [7]:
delete = ["accept_language", "browser_language"]

train_part1 = train_part1.drop(delete)

In [8]:
train_part1 = train_part1.with_columns(pl.col("event_dttm").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S")\
                                 .alias("event_dttm")).with_columns(pl.col("event_dttm").dt.hour().alias("Hour"))

In [9]:
train_part1 = train_part1.sort("event_dttm")
train_part1 = train_part1.drop("event_dttm")

In [10]:
cat_features = [
    'mcc_code', 'event_desc',
    'timezone', 'operating_system_type', 'device_system_version',
    'screen_size', 'battery'
]

for i in cat_features:
    train_part1 = train_part1.with_columns(pl.col(i).fill_null('missing'))

train_part1 = train_part1.with_columns(pl.concat_str([pl.col("operating_system_type"), pl.col("device_system_version"), 
    pl.col("screen_size"), pl.col("battery")], separator="_").alias("device_info"))

In [11]:
train_part1 = train_part1.join(pre_agg, on="customer_id", how="left")
train_part1 = train_part1.drop("customer_id")

In [12]:
event_id = train_part1["event_id"].to_pandas()
train_part1 = train_part1.drop("event_id")

In [13]:
train_part1

event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,Hour,device_info,mean_amt_pre,median_amt_pre,std_amt_pre,ops_pre,most_common_hour_pre
i32,str,i32,i32,f64,i32,str,i32,str,i64,str,str,str,str,i32,i32,i32,i32,i8,str,f64,f64,f64,u32,i8
8,"""122""",2,0,null,null,"""missing""",null,"""missing""",126121010775326,"""missing""","""missing""","""missing""","""missing""",null,null,null,null,0,"""missing_missing_missing_missin…",null,null,null,null,null
7,"""56""",3,4,null,null,"""missing""",null,"""16""",124214045774300,"""9""","""99%""","""missing""","""missing""",null,null,0,null,0,"""9_missing_missing_99%""",null,null,null,null,null
14,"""75""",0,5,85864.0,0,"""19""",null,"""missing""",null,"""missing""","""missing""","""missing""","""missing""",null,null,null,null,0,"""missing_missing_missing_missin…",null,null,null,null,null
14,"""75""",0,5,44216.0,0,"""4""",null,"""missing""",null,"""missing""","""missing""","""missing""","""missing""",null,null,null,null,0,"""missing_missing_missing_missin…",105069.186896,39636.0,285422.72566,1576,1
2,"""88""",0,5,null,0,"""10""",3,"""missing""",null,"""missing""","""missing""","""missing""","""missing""",null,null,null,null,0,"""missing_missing_missing_missin…",534833.0,100845.0,1.0650e6,444,6
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7,"""56""",3,4,null,null,"""missing""",null,"""42""",123818908925253,"""6""","""not available""","""missing""","""missing""",null,null,0,null,23,"""6_missing_missing_not availabl…",null,null,null,null,null
8,"""122""",2,0,null,null,"""missing""",null,"""missing""",126318579340682,"""missing""","""missing""","""missing""","""missing""",null,null,null,null,23,"""missing_missing_missing_missin…",null,null,null,null,null
14,"""75""",0,5,65121.0,0,"""4""",3,"""missing""",null,"""missing""","""missing""","""missing""","""missing""",null,null,null,null,23,"""missing_missing_missing_missin…",null,null,null,null,null


Получили обработанный набор данных, теперь можно переходить прогнозированию.

In [ ]:
model = CatBoostClassifier()
model.load_model('../Models/Model_PC_4.2.cbm')

CatBoostClassifier(class_names=[0, 1], class_weights=[1, 100], depth=6, iterations=1000, l2_leaf_reg=10, learning_rate=0.015, loss_function='Logloss', max_ctr_complexity=5, od_wait=100, one_hot_max_size=20, verbose=0)

In [15]:
predict = model.predict(train_part1)

In [16]:
y_pred_proba = model.predict_proba(train_part1)[:, 1]
y_pred_proba

array([0.04271705, 0.273124  , 0.39267711, ..., 0.26388984, 0.3011977 ,
       0.20319748], shape=(633683,))

In [17]:
y_pred_proba = pd.Series(y_pred_proba, name="predict")
y_pred_proba

0         0.042717
1         0.273124
2         0.392677
3         0.221206
4         0.479791
            ...   
633678    0.217480
633679    0.045664
633680    0.263890
633681    0.301198
633682    0.203197
Name: predict, Length: 633683, dtype: float64

In [18]:
event_id = pd.Series(event_id, name="event_id")

In [19]:
submit = pd.concat([event_id, y_pred_proba], names=["event_id", "predict"], axis=1)

In [20]:
submit

,event_id,predict
0,125339330014816,0.042717
1,124978549756126,0.273124
2,126198320503253,0.392677
3,125390866897300,0.221206
4,123879040110074,0.479791
...,...,...
633678,126146783971142,0.217480
633679,123999297165929,0.045664
633680,123561209857910,0.263890
633681,125674336254442,0.301198


In [21]:
submit.to_csv("submit.csv", index=False)